# Iron Condor Backtest Demo

This notebook demonstrates a simple iron condor strategy using the NSE options data pipeline.
We'll load sample data, construct an iron condor for a given expiry, and calculate PnL, win rate, and max drawdown.

> **Note**: This is a simplified example for demonstration purposes. In practice, you would want to refine the strategy, consider transaction costs, and perform more rigorous analysis.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Set plotting style
plt.style.use('seaborn-v0_8')
sns = __import__('seaborn')
sns.set_palette("husl")

In [ ]:
# Load sample data
df = pd.read_csv('../data/sample_data.csv')
print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
# Convert timestamp to datetime if needed
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])

# For simplicity, we'll assume the data contains columns:
# ['timestamp', 'underlying', 'strike', 'option_type', 'bid', 'ask', 'iv', 'volume', 'open_interest']
# We'll filter for a specific expiry (example: nearest weekly expiry)
# In real data, you would have an expiry column.
# For demo, we'll just use the whole dataset and assume it's for a single expiry.

# Let's examine the columns
print("Columns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

In [ ]:
# For demonstration, we'll create a simplified option chain snapshot.
# In a real backtest, you would iterate over time and adjust positions.

# Let's assume we have data for a single day and we want to sell an iron condor:
# - Sell OTM put (lower strike)
# - Buy OTM put (further lower strike) for protection
# - Sell OTM call (higher strike)
# - Buy OTM call (further higher strike) for protection

# We'll need to select strikes based on the underlying price and desired width.

# Example parameters
underlying_price = df['underlying'].iloc[0] if 'underlying' in df.columns else 18000
print(f"Underlying price: {underlying_price}")

# Choose strikes: typically based on delta or standard deviations.
# For demo, we'll pick round numbers.
put_sell_strike = underlying_price - 200  # 200 points OTM put
put_buy_strike = put_sell_strike - 100   # further OTM for protection
call_sell_strike = underlying_price + 200 # 200 points OTM call
call_buy_strike = call_sell_strike + 100  # further OTM for protection

print(f"Put Sell Strike: {put_sell_strike}")
print(f"Put Buy Strike: {put_buy_strike}")
print(f"Call Sell Strike: {call_sell_strike}")
print(f"Call Buy Strike: {call_buy_strike}")

In [ ]:
# Filter options for the selected strikes and types
def get_option(df, strike, option_type):
    # Assuming columns: strike, option_type, bid, ask
    # We'll take the mid price as (bid + ask)/2
    mask = (df['strike'] == strike) & (df['option_type'] == option_type)
    if not mask.any():
        # If exact strike not found, find closest
        idx = (df['strike'] - strike).abs().idxmin()
        strike = df.loc[idx, 'strike']
        option_type = df.loc[idx, 'option_type']
        mask = (df['strike'] == strike) & (df['option_type'] == option_type)
    row = df.loc[mask].iloc[0]
    mid_price = (row['bid'] + row['ask']) / 2
    return {
        'strike': row['strike'],
        'option_type': row['option_type'],
        'bid': row['bid'],
        'ask': row['ask'],
        'mid': mid_price,
        'iv': row.get('iv', np.nan)
    }

put_sell = get_option(df, put_sell_strike, 'PE')
put_buy = get_option(df, put_buy_strike, 'PE')
call_sell = get_option(df, call_sell_strike, 'CE')
call_buy = get_option(df, call_buy_strike, 'CE')

print("\nPut Sell:", put_sell)
print("\nPut Buy:", put_buy)
print("\nCall Sell:", call_sell)
print("\nCall Buy:", call_buy)


In [ ]:
# Calculate initial credit received (for iron condor, we sell the inner strikes and buy the outer strikes)
# Credit = (put_sell.mid + call_sell.mid) - (put_buy.mid + call_buy.mid)
credit = (put_sell['mid'] + call_sell['mid']) - (put_buy['mid'] + call_buy['mid'])
print(f"Net credit received: {credit:.2f}")

# Maximum profit = credit (if underlying stays between inner strikes at expiry)
max_profit = credit

# Maximum loss = width of one spread - credit
# Width of put spread = put_sell.strike - put_buy.strike
# Width of call spread = call_buy.strike - call_sell.strike (should be same)
put_width = put_sell['strike'] - put_buy['strike']
call_width = call_buy['strike'] - call_sell['strike']
max_loss = min(put_width, call_width) - credit
print(f"Put spread width: {put_width}")
print(f"Call spread width: {call_width}")
print(f"Maximum loss per spread: {max_loss:.2f}")

# Break-even points
lower_breakeven = put_sell['strike'] - credit
upper_breakeven = call_sell['strike'] + credit
print(f"Lower break-even: {lower_breakeven:.2f}")
print(f"Upper break-even: {upper_breakeven:.2f}")

## Simulating PnL over time

For demonstration, we'll assume we hold the position until expiry and the underlying price at expiry is known.
In a real backtest, you would simulate daily PnL based on price movements.

We'll create a simple scenario: underlying price at expiry is normally distributed around the current price.
We'll simulate 1000 possible underlying prices at expiry and calculate the PnL for each.

> This is a Monte Carlo simulation for illustration only.


In [ ]:
# Monte Carlo simulation of underlying price at expiry
np.random.seed(42)
simulated_prices = np.random.normal(loc=underlying_price, scale=underlying_price*0.05, size=1000)  # 5% volatility

def calculate_pnl(price):
    # Iron condor PnL at expiry
    # Put spread: max(0, put_sell_strike - price) - max(0, put_buy_strike - price)
    put_pnl = max(0, put_sell['strike'] - price) - max(0, put_buy['strike'] - price)
    # Call spread: max(0, price - call_sell_strike) - max(0, price - call_buy_strike)
    call_pnl = max(0, price - call_sell['strike']) - max(0, price - call_buy['strike'])
    # Total PnL = credit received - (put_pnl + call_pnl)  (since we sold spreads)
    # Actually, we sold the inner strikes and bought the outer strikes, so we receive credit and pay for spreads.
    # The PnL from the put spread is: -max(0, put_sell_strike - price) + max(0, put_buy_strike - price)  (because we sold put_sell and bought put_buy)
    # Similarly for call: -max(0, price - call_sell_strike) + max(0, price - call_buy_strike)
    # Then add the credit.
    put_spread_pnl = -max(0, put_sell['strike'] - price) + max(0, put_buy['strike'] - price)
    call_spread_pnl = -max(0, price - call_sell['strike']) + max(0, price - call_buy['strike'])
    total_pnl = credit + put_spread_pnl + call_spread_pnl
    return total_pnl

pnl_series = np.array([calculate_pnl(p) for p in simulated_prices])

print(f"Mean PnL: {np.mean(pnl_series):.2f}")
print(f"Median PnL: {np.median(pnl_series):.2f}")
print(f"Std PnL: {np.std(pnl_series):.2f}")
print(f"Win rate (PnL > 0): {np.mean(pnl_series > 0)*100:.2f}%")
print(f"Max drawdown: {np.min(pnl_series):.2f}")
print(f"95% VaR: {np.percentile(pnl_series, 5):.2f}")

In [ ]:
# Plot PnL distribution
plt.figure(figsize=(10, 6))
plt.hist(pnl_series, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', label='Break-even')
plt.axvline(x=np.mean(pnl_series), color='green', linestyle='-', label=f'Mean: {np.mean(pnl_series):.2f}')
plt.xlabel('PnL per lot')
plt.ylabel('Frequency')
plt.title('Distribution of Iron Condor PnL at Expiry (Monte Carlo Simulation)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
*This demo uses simulated data for illustrative purposes. Replace the sample data with real processed data from the pipeline for actual backtesting.*
